# 01 — Explore the Data

This notebook explores the BRONZE data layer:
- Salesforce CRM: Opportunities, Vehicles (Assets), Defects, Deliveries
- Odos Telemetry: Raw GPS/battery/speed events
- VIN overlap between the two systems


In [ ]:
-- Set context
SET user_db = 'HYDRAB_HOL_' || CURRENT_USER();
USE DATABASE IDENTIFIER($user_db);
USE SCHEMA BRONZE;
USE WAREHOUSE HYDRAB_HOL_WH;

## Salesforce: Sales Pipeline

In [ ]:
-- Top opportunities by value
SELECT "Name", "StageName", "Amount", "CloseDate", "Region__c"
FROM OPPORTUNITY
WHERE "Amount" > 1000000
ORDER BY "Amount" DESC
LIMIT 20;

In [ ]:
-- Pipeline by stage
SELECT "StageName", 
       COUNT(*) as deals, 
       SUM("Amount") as total_value,
       ROUND(AVG("Amount"), 0) as avg_deal
FROM OPPORTUNITY
GROUP BY "StageName"
ORDER BY total_value DESC;

## Salesforce: Vehicle Fleet

In [ ]:
-- Fleet composition by model
SELECT "Product_Name__c" as model, 
       COUNT(*) as vehicles,
       COUNT(DISTINCT "Account_Name__c") as customers
FROM ASSET
GROUP BY "Product_Name__c"
ORDER BY vehicles DESC
LIMIT 10;

## Odos Telemetry: Raw Events

In [ ]:
-- Sample telemetry events (raw JSON)
SELECT *
FROM ODOS_EVENTS
LIMIT 5;

In [ ]:
-- Parse key signals from Odos JSON
SELECT 
  RAW:vin::STRING as vin,
  RAW:timestamp::TIMESTAMP as event_time,
  RAW:signals:battery_soc::FLOAT as battery_soc_pct,
  RAW:signals:speed::FLOAT as speed_kmh,
  RAW:signals:battery_temperature::FLOAT as temp_celsius,
  RAW:signals:latitude::FLOAT as lat,
  RAW:signals:longitude::FLOAT as lng
FROM ODOS_EVENTS
LIMIT 20;

## The VIN Join: Connecting CRM to Telemetry

In [ ]:
-- How many VINs overlap between Salesforce and Odos?
SELECT COUNT(DISTINCT a."Chassis_Number__c") as matching_vins
FROM ASSET a
INNER JOIN (
  SELECT DISTINCT RAW:vin::STRING as vin 
  FROM ODOS_EVENTS
) o ON a."Chassis_Number__c" = o.vin;

In [ ]:
-- Joined view: vehicle details + latest telemetry
SELECT 
  a."Chassis_Number__c" as vin,
  a."Product_Name__c" as model,
  a."Account_Name__c" as customer,
  o.battery_soc_pct,
  o.speed_kmh,
  o.event_time
FROM ASSET a
INNER JOIN (
  SELECT 
    RAW:vin::STRING as vin,
    RAW:signals:battery_soc::FLOAT as battery_soc_pct,
    RAW:signals:speed::FLOAT as speed_kmh,
    RAW:timestamp::TIMESTAMP as event_time,
    ROW_NUMBER() OVER (PARTITION BY RAW:vin::STRING ORDER BY RAW:timestamp DESC) as rn
  FROM ODOS_EVENTS
) o ON a."Chassis_Number__c" = o.vin AND o.rn = 1
ORDER BY o.battery_soc_pct ASC
LIMIT 15;

## Explore Complete!

Key findings:
- 2,492 VINs match between Salesforce and Odos
- Telemetry includes SOC, speed, temperature, GPS coordinates
- Sales pipeline has 1,056 opportunities worth £2.5B+

**Next:** Open `03_build_gold.ipynb` to build the analytics layer.